# exp2 7B Phase 0 - self-contained (no GitHub token, no Drive mount)

Generated 2026-08-16 from `experiment 2/colab/00_setup_schema_audit.ipynb`.
**Only the clone cell was replaced**; every pre-registered Phase-0 cell below is
byte-for-byte the committed version.

Config actually loaded: `exp2_colab_config_mvp.json` (MVP scope fork registered
2026-08-16 - model stays Qwen2.5-7B, scope is cut on the update axis).

## How to run
1. Runtime -> Change runtime type -> **A100 GPU** -> Save
2. Runtime -> Run all
3. Walk away. Phase 0 on a 7B base model is expected to take well over an hour,
   most of it generation.

This runs Phase 0 only (contract re-verification, token audit, split freeze,
Gate C0 memory calibration, sparse-reward preflight, 2-update smoke). **It does
not start Stage A.**

## Why this exists instead of the normal notebook 00

`00_setup_schema_audit.ipynb` clones the private repo with a PAT from Colab
Secrets. That PAT is **broken** — verified 2026-08-16, the clone fails with
`remote: Write access to repository not granted` / HTTP 403. Colab's own GitHub
integration (OAuth) reads the private repo fine, so this notebook is opened
through that instead and carries its source inline, touching no PAT anywhere.

**The embedded blob is a SNAPSHOT.** If `experiment 2/src/`,
`experiment 2/vendor/`, either config, `requirements.txt`, or
`eaaj-pilot/src/{metrics,callbacks}.py` changes, this notebook is stale.
Regenerate it with `scripts/build_7b_selfcontained.py` and re-commit; do not
hand-edit the blob. Delete this notebook once the PAT is fixed.

## Deviation logged up front
The config registers "L4 first, escalate on Gate C0". This notebook's GPU gate
requires >= 35 GiB and therefore goes straight to A100. Reason: on 2026-08-16 the
0.5B track at this identical group-8 geometry was measured needing ~27 GiB and
OOM'd on a 22 GiB L4. A 7B base cannot fit where 0.5B did not. Trying L4 first
would spend a full model-download cycle to learn something already measured,
which defeats the "cheaper compute-unit draw" rationale L4-first was registered
for.


In [ ]:
#@title 1 GPU gate - refuse to continue on unsuitable hardware
# Deliberately does NOT import torch. torch imports numpy, and this notebook
# later installs a pinned numpy that is a major version ahead of Colab's; if
# numpy is already resident when that install lands, every downstream import
# dies ("cannot import name '_center' from 'numpy._core.umath'") and the only
# cure is a kernel restart, which turns Run all into a two-pass process that
# needs a human to press it again. On 2026-08-16 that cost two runtimes to idle
# reclamation. Querying nvidia-smi in a subprocess keeps this cell's fail-fast
# value without touching the numpy that cell 3 is about to replace.
import subprocess, sys

_q = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True)
if _q.returncode != 0 or not _q.stdout.strip():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100 GPU -> Save")

_name, _mem_mib, _cap = [f.strip() for f in _q.stdout.strip().splitlines()[0].split(",")]
total_gb = int(_mem_mib) / 1024
cap_major = int(float(_cap))
print(f"GPU          : {_name}")
print(f"compute cap  : {_cap}")
print(f"total memory : {total_gb:.1f} GiB")

problems = []
if cap_major < 8:
    problems.append(
        f"Architecture too old (cap {_cap}). The recipe uses bfloat16, which "
        f"needs Ampere (8.0) or newer. T4/V100 will not work.")
if total_gb < 35:
    problems.append(
        f"Only {total_gb:.1f} GiB of VRAM. Qwen2.5-7B in bf16 is ~15 GiB of weights "
        f"before any group-8 generation state. Measured evidence from 2026-08-16: "
        f"the 0.5B track at this same group-8 geometry already needed ~27 GiB and "
        f"OOM'd on a 22 GiB L4. Use A100.")

if problems:
    print("\nUnsuitable GPU:")
    for p in problems:
        print("  -", p)
    raise SystemExit("GPU gate failed")
print("\nGPU gate passed (bf16 support is re-confirmed against torch in cell 4)")


In [ ]:
#@title 2 Unpack embedded source (no GitHub token required)
# This notebook carries the repo files it needs as a gzip+base64 blob instead of
# cloning the private repo. Same trick the v9 4070 probe used on 2026-08-16, for
# the same reason: the private-repo clone needs a PAT, and getting that PAT's
# fine-grained permissions right repeatedly failed (and leaked the token into
# cell output twice before it was sanitized). No token means neither failure mode
# can happen. The layout below reproduces the sibling-directory structure
# src/pipeline.py depends on (EXP2_ROOT/.. must contain eaaj-pilot/src).
import base64, gzip, io, os, sys, json, tarfile
from pathlib import Path

_B64 = """
...
"""

REPO_DIR = "/content/RLVR"
os.makedirs(REPO_DIR, exist_ok=True)
_raw = gzip.decompress(base64.b64decode("".join(_B64.split())))
with tarfile.open(fileobj=io.BytesIO(_raw), mode="r") as _tar:
    _tar.extractall(REPO_DIR)

EXP2_DIR = f"{REPO_DIR}/experiment 2"
os.makedirs(f"{EXP2_DIR}/data", exist_ok=True)
for _p in sorted(Path(REPO_DIR).rglob("*")):
    if _p.is_file():
        print(" ", _p.relative_to(REPO_DIR))
print("\nunpacked into", REPO_DIR)


In [ ]:
#@title 3 Install pinned dependencies, keeping the resident numpy (2-4 min)
# DEVIATION LOGGED 2026-08-16: requirements.txt pins numpy==2.3.5 to mirror the
# local env, but Colab's kernel has numpy RESIDENT from interpreter startup
# (2.0.2 today) - before any user cell runs. Upgrading numpy across versions
# under a live kernel breaks every later import ("cannot import name '_center'
# from 'numpy._core.umath'"), and the only cure is a kernel restart, which makes
# Run all two-pass and cost three runtimes to idle reclamation on 2026-08-16.
# So numpy is deliberately HELD at the resident version via a pip constraint.
# The scientific manifest (trl/transformers/datasets/accelerate/peft/
# bitsandbytes/pylatexenc, checked in cell 4) is installed exactly as pinned;
# numpy is not part of that contract and its exact patch version does not enter
# GRPO semantics. torch is NOT reinstalled - Colab's CUDA build is kept.
import importlib.metadata as _md

_resident_numpy = _md.version("numpy")
_req = open("/content/RLVR/experiment 2/requirements.txt").read().splitlines()
_kept = [l for l in _req if not l.strip().lower().startswith("numpy")]
open("/tmp/req_no_numpy.txt", "w").write("\n".join(_kept))
open("/tmp/constraints.txt", "w").write(f"numpy=={_resident_numpy}\n")
print(f"holding numpy at resident {_resident_numpy}; installing the rest as pinned")

%pip install -q -r /tmp/req_no_numpy.txt -c /tmp/constraints.txt
print("\nInstall finished.")


In [ ]:
#@title 3b Assert the kernel is clean (must be a no-op)
# With the cell-1 reorder nothing imports numpy before cell 3 installs it, so
# the loaded and installed versions must agree on the first pass and Run all
# completes without a restart. If this ever fires, the ordering invariant broke:
# something above imported numpy (directly, or via torch/pandas) before the
# install. Fix the ordering rather than adding a restart - a restart makes Run
# all two-pass, which needs an operator present and cost two runtimes to idle
# reclamation on 2026-08-16.
import importlib.metadata as _md
import numpy as _np

_installed = _md.version("numpy")
if _np.__version__ != _installed:
    raise SystemExit(
        f"ORDERING BROKEN: numpy {_np.__version__} was already resident before "
        f"the install put {_installed} on disk. Something above cell 3 imports "
        "numpy. Restarting would work but re-introduces the two-pass problem - "
        "fix the import order instead.")
print(f"kernel clean: numpy loaded == installed == {_installed}, single pass OK")


In [ ]:
#@title 4 Environment check - versions must match the pinned manifest
import importlib.metadata as md
import sys
import torch

EXPECTED = {"trl": "1.6.0", "transformers": "5.13.0", "datasets": "5.0.0",
            "accelerate": "1.14.0", "peft": "0.15.2", "bitsandbytes": "0.49.2",
            "pylatexenc": "2.10"}

print("python", ".".join(map(str, sys.version_info[:3])))
print("torch ", torch.__version__, f"(CUDA {torch.version.cuda})  <- Colab preinstalled")
print()
mismatched = []
for pkg, want in EXPECTED.items():
    try:
        got = md.version(pkg)
    except Exception:
        got = "MISSING"
    if got != want:
        mismatched.append(pkg)
    print(f"  {'ok ' if got == want else 'BAD'} {pkg:<14} want {want:<10} got {got}")

if mismatched:
    print("\nVersion mismatch:", ", ".join(mismatched))
    print("TRL version drift can silently change GRPO semantics. Report before trusting results.")
else:
    print("\nAll pinned packages match the manifest.")
import numpy as _np_v
print(f"numpy held at resident {_np_v.__version__} (deviation logged in cell 3; not part of the pinned manifest)")

# Import the bundled modules now, in-process, so a missing dependency surfaces in
# seconds rather than after the 7B download.
sys.path.insert(0, "/content/RLVR/experiment 2")
import src.guru_data, src.guru_reward, src.pipeline  # noqa: F401
from vendor.reasoning360_reward_score import codeio, naive_dapo  # noqa: F401
print("Import smoke test passed: bundled data/reward/pipeline modules load cleanly.")


In [ ]:
#@title 5 Pre-download the model and dataset (7B is ~15 GB - several minutes)
# Not strictly required (unlike the 4070 track, this code does not force
# local_files_only), but downloading here isolates a network failure from a
# training failure, and gives one clean progress bar instead of a stall in the
# middle of Phase 0.
import json, subprocess, sys, time

CFG = json.load(open("/content/RLVR/experiment 2/exp2_colab_config_mvp.json"))
MODEL_ID = CFG["model_id"]
MODEL_REVISION = CFG["model_revision"]
DS_SOURCE = CFG["dataset"]["source"]
DS_REVISION = CFG["dataset"]["revision"]
print(f"model  : {MODEL_ID} @ {MODEL_REVISION}")
print(f"dataset: {DS_SOURCE} @ {DS_REVISION}")

code = "from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM\n"
code += "from huggingface_hub import snapshot_download\n"
code += f"AutoTokenizer.from_pretrained({MODEL_ID!r}, revision={MODEL_REVISION!r})\n"
code += f"AutoConfig.from_pretrained({MODEL_ID!r}, revision={MODEL_REVISION!r})\n"
code += f"snapshot_download({MODEL_ID!r}, revision={MODEL_REVISION!r})\n"
code += (f"snapshot_download({DS_SOURCE!r}, repo_type='dataset', "
         f"revision={DS_REVISION!r})\n")

for attempt in range(1, 6):
    print(f"Attempt {attempt}/5 ...", flush=True)
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    if r.returncode == 0:
        print("Download successful. Model weights and dataset are cached.")
        break
    print("Failed:", (r.stderr or "").strip().splitlines()[-1:] or "(no stderr)")
    if attempt < 5:
        time.sleep(15)
else:
    raise SystemExit("Could not download from Hugging Face after 5 attempts.")


## Phase 0 - pre-registered cells below are unmodified

In [ ]:
import json, os, sys
from pathlib import Path

# Replaces the original notebook's private-repo clone. The source is already on
# disk from cell 2; everything below defines exactly the same globals the rest of
# this notebook expects, so the pre-registered cells that follow are unmodified.
REPO_DIR = "/content/RLVR"
EXP2_DIR = f"{REPO_DIR}/experiment 2"
if EXP2_DIR not in sys.path:
    sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path - pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

# MVP scope fork - see EXPERIMENT_2_COLAB_MVP_AMENDMENT.md. This is the ONE
# knob: switch it back to 'exp2_colab_config.json' if and only if the Phase-0
# promotion gate passes. Never switch it mid-run.
CONFIG_NAME = 'exp2_colab_config_mvp.json'
CONFIG = json.load(open(f'{EXP2_DIR}/{CONFIG_NAME}'))
DATA_DIR = Path(EXP2_DIR) / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID, MODEL_REVISION = CONFIG['model_id'], CONFIG['model_revision']
DATASET_REVISION = CONFIG['dataset']['revision']
print('config loaded:', CONFIG['experiment'])
print('status       :', CONFIG['status'])
print('model        :', MODEL_ID, '| variant:', CONFIG.get('model_variant'))
print('stage_a      : max_steps', CONFIG['stage_a']['max_steps'],
      '| checkpoints', CONFIG['stage_a']['checkpoint_steps'],
      '| group', CONFIG['stage_a']['num_generations'])


In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

## Step 1-3 — load the confirmed contract, spot-check rows

`load_all_records` renders every prompt through THIS model's chat template and computes token counts with THIS model's tokenizer — the field names and file paths are pinned/confirmed, but the counts below are still real numbers for this model, not copy-pasted from the 0.5B track.

In [ ]:
math_rows, sim_rows = guru_data.load_all_records(
    MODEL_ID, MODEL_REVISION, DATASET_REVISION,
    stage_a_prompt_suffix=None)
print('Math (stage A) rows:', len(math_rows))
print('Simulation/CodeIO (stage B) rows:', len(sim_rows))
print()
print('--- sample Math row ---')
sample = math_rows[0]
print({k: (v[:300] if isinstance(v, str) else v) for k, v in sample.items() if k != 'extra_info'})
print()
print('--- sample Simulation row ---')
sample = sim_rows[0]
print({k: (v[:300] if isinstance(v, str) else v) for k, v in sample.items() if k != 'extra_info'})

**Sanity check before continuing:** do the two counts above look like the confirmed audit's `stage_a_count`/`stage_b_count` (`data/guru_schema_audit.json`), and does each sample row have a real rendered prompt (not an empty string or a raw message-list repr) and a real ground_truth? If not, stop and investigate — do not proceed on a loader that silently produced garbage.

## Step 4 — token-length audit under this model's tokenizer (GATE 0a re-verification)

In [ ]:
audit_a_raw = guru_data.token_stats(math_rows)
audit_b_raw = guru_data.token_stats(sim_rows)

# GATE 0a - DEVIATION LOGGED 2026-08-16 (operator decision, Aaron; flagged to
# Tommy, not silently decided - see
# FINDING_GATE_0A_MEASURES_THE_WRONG_POPULATION.md). The gate now audits the
# token_filter_max-ELIGIBLE population - the rows training actually uses,
# selected by the identical filter expression build_exp2_splits applies -
# instead of the raw pool. The raw-pool reading is unsatisfiable on any
# hardware (p95=1407 is a dataset property; an A100 80GB reproduced the
# WIN4070 number exactly), and the configs' own phase0a_note shows the
# filtered population was what this gate was always meant to check.
# Registered numbers UNCHANGED: threshold 1024, token_filter_max 640.
math_eligible = [r for r in math_rows
                 if r['prompt_tokens'] <= CONFIG['stage_a']['token_filter_max']]
sim_eligible = [r for r in sim_rows
                if r['prompt_tokens'] <= CONFIG['stage_b']['token_filter_max']]
audit_a = guru_data.token_stats(math_eligible)
audit_b = guru_data.token_stats(sim_eligible)
print('stage A (Math) raw      :', audit_a_raw)
print('stage B (Sim)  raw      :', audit_b_raw)
print('stage A (Math) eligible :', audit_a)
print('stage B (Sim)  eligible :', audit_b)

# cross-track verification: the raw audit must still match the confirmed
# data/token_length_audit.json numbers (0.5B track) - if it drifts, the
# loader or dataset changed and NOTHING below is trustworthy.
_expected_raw_b = {'n': 3730, 'p50': 700.0, 'p95': 1407.0, 'max': 1949}
_raw_ok = all(abs(audit_b_raw[k] - v) < 1.5 for k, v in _expected_raw_b.items())
print('raw stage-B audit matches WIN4070 reference:', _raw_ok)
if not _raw_ok:
    raise SystemExit('GATE 0a STOP: raw audit does not match the confirmed '
                     'reference - investigate the loader before anything else.')

gate_0a_threshold = CONFIG['gates']['phase0a_stage_b_p95_prompt_tokens_max']
if audit_b['p95'] > gate_0a_threshold:
    raise SystemExit(
        f"GATE 0a STOP: eligible stage-B p95={audit_b['p95']} > {gate_0a_threshold}. "
        'Escalate GPU tier - do not shrink the batch to force a fit.')
print(f"GATE 0a: PASS on the eligible population (p95={audit_b['p95']} <= "
      f"{gate_0a_threshold}; raw p95={audit_b_raw['p95']} recorded above, not gated)")


## Step 5 — freeze splits (train/eval/probe), model- and geometry-specific

In [ ]:
splits = guru_data.build_exp2_splits(
    MODEL_ID, MODEL_REVISION,
    stage_a_token_limit=CONFIG['stage_a']['token_filter_max'],
    stage_b_token_limit=CONFIG['stage_b']['token_filter_max'],
    stage_b_eval_questions=CONFIG['stage_b']['eval_questions'],
    n_probe=CONFIG['measurement']['probe_questions'],
    dataset_revision=DATASET_REVISION, seed=CONFIG['seed'],
    out_name='exp2_colab_splits.json')
print('stage_a_train:', len(splits['stage_a_train_ids']),
      '| stage_b_train:', len(splits['stage_b_train_ids']),
      '| stage_b_eval:', len(splits['stage_b_eval_ids']),
      '| probe:', splits['probe_actual'], '/', splits['probe_requested'])
if 'probe_shortfall_note' in splits:
    print('WARNING:', splits['probe_shortfall_note'])

## Gate C0 — GPU memory calibration at the REAL group-8 geometry

This is the first time group 8 will run to completion anywhere in this project (the WIN4070 track's own group-8 attempt OOM'd before finishing its smoke). Escalate tier if it doesn't fit — do not shrink `num_generations`/batch below the config to force an L4 fit.

In [ ]:
sa = CONFIG['stage_a']
smoke_ds = guru_data.to_hf_dataset(math_rows[:8])

gate_c0 = pipeline.gate_c0_memory_probe(
    MODEL_ID, CONFIG['peft'], smoke_ds, sa['reward_mode'],
    num_generations=sa['num_generations'],
    per_device_batch=sa['per_device_train_batch_size'],
    grad_accum=sa['gradient_accumulation_steps'],
    max_completion_length=sa['max_completion_length'],
    device='cuda', min_headroom_pct=CONFIG['gates']['gate_c0_memory_headroom_min_pct'],
    learning_rate=sa['learning_rate'])
print(gate_c0)
if not gate_c0['gate_pass']:
    print('Gate C0: escalate to A100 (switch the Colab runtime, then re-run this cell). '
          'This was the EXPECTED outcome per the plan (§1, GPU tier row) - not a surprise.')
else:
    print('Gate C0: PASS on current tier.')

## Phase 0 step 7-8 — smoke test + tightened sparse-reward preflight (GATE 0b)

16 frozen Stage-A prompts x 8 generations, 8 frozen Stage-B prompts x 8 generations. STOP unless >=2 groups have variable COMBINED reward on EACH stage; exact-channel variance is tracked and reported separately (`FINDING_GROUP_SIZE_REWARD_VARIANCE.md` — combined variance alone overstates how much of the group-8 gain is real reasoning signal vs. format-shaping noise).

In [ ]:
model, tokenizer = pipeline.build_peft_model(MODEL_ID, CONFIG['peft'], device='cuda')

stage_a_preflight_rows = [
    r for r in math_rows if r['id'] in set(splits['stage_a_train_ids'])
][:CONFIG['gates']['phase0b_stage_a_preflight_prompts']]
stage_b_preflight_rows = [
    r for r in sim_rows if r['id'] in set(splits['stage_b_train_ids'])
][:CONFIG['gates']['phase0b_stage_b_preflight_prompts']]

preflight_a = pipeline.guru_sparse_reward_preflight(
    model, tokenizer, stage_a_preflight_rows, sa['reward_mode'],
    num_generations=sa['num_generations'],
    min_variable_groups=CONFIG['gates']['phase0b_min_variable_groups'])
sb = CONFIG['stage_b']
preflight_b = pipeline.guru_sparse_reward_preflight(
    model, tokenizer, stage_b_preflight_rows, sb['reward_mode'],
    num_generations=sb['num_generations'],
    min_variable_groups=CONFIG['gates']['phase0b_min_variable_groups'])

for name, pf in [('Stage A (Math)', preflight_a), ('Stage B (Simulation)', preflight_b)]:
    print(f"{name}: combined-variable groups {pf['groups_with_combined_variance']}/{pf['n_prompts']}, "
          f"exact-variable groups {pf['groups_with_exact_variance']}/{pf['n_prompts']}, "
          f"has_grpo_signal={pf['has_grpo_signal']}")

if not (preflight_a['has_grpo_signal'] and preflight_b['has_grpo_signal']):
    raise SystemExit(
        'GATE 0b STOP: fewer than the required variable groups on at least one stage. '
        'Preserve this preflight result and ask the team. Do not add extra shaping reward '
        'beyond the registered exact_plus_boxed_format_0.1 mode; if the failure looks like '
        'a format-compliance problem specifically, consider the Instruct fallback (config '
        "'model_variant_contingency') and log the deviation - don't silently switch.")
print('GATE 0b: PASS on both stages')

**Format-following check (Base vs Instruct contingency, plan §1/§8 item 4):** scan `preflight_a['groups'][*]['completion_tails']` above for repeated failure to emit a well-formed `\boxed{}`. If most completions never attempt the format, that is the specific signal the WIN4070 track's switch to Instruct was responding to at 0.5B scale — flag it before spending Phase 1 compute on a base model that can't be scored.

In [ ]:
model, tokenizer = pipeline.build_peft_model(MODEL_ID, CONFIG['peft'], device='cuda')
smoke_a = guru_data.to_hf_dataset(stage_a_preflight_rows[:8])
smoke_b = guru_data.to_hf_dataset(stage_b_preflight_rows[:8])

for label, ds, mode, geom in [
    ('stage A smoke', smoke_a, sa['reward_mode'], sa),
    ('stage B smoke', smoke_b, sb['reward_mode'], sb),
]:
    from trl import GRPOConfig, GRPOTrainer
    cfg = GRPOConfig(
        output_dir=f'/tmp/exp2_smoke_{label.replace(" ", "_")}', seed=42, max_steps=2,
        learning_rate=geom['learning_rate'], per_device_train_batch_size=geom['per_device_train_batch_size'],
        gradient_accumulation_steps=geom['gradient_accumulation_steps'], num_generations=geom['num_generations'],
        beta=geom['beta'], max_completion_length=geom['max_completion_length'],
        bf16=True, optim='paged_adamw_8bit', gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=1, save_strategy='no', report_to='none')
    trainer = GRPOTrainer(model=model, args=cfg, train_dataset=ds,
                          reward_funcs=guru_reward.select_reward_fn(mode), processing_class=tokenizer)
    trainer.train()
    print(label, 'completed 2/2 smoke updates OK')

## Commit reminder

Commit `data/exp2_colab_splits.json` with message prefix `exp2-colab:`. Log this phase's wall time, GPU tier, and Colab compute-unit cost in `eaaj-pilot/compute_log.md` before moving to notebook 01.

In [ ]:
#@title Persist Phase-0 artifacts (Drive if possible, else base64 in output)
# /content is ephemeral (lesson from the v9 probe: the runtime was recycled
# overnight and every artifact vanished). The frozen splits and audits below
# must reach the repo, and the Colab PAT is broken, so they go home via Drive
# or, failing that, inline base64 that gets transcribed from this output.
import base64, gzip, io, os, tarfile

SRC = "/content/RLVR/experiment 2/data"
names = sorted(n for n in os.listdir(SRC) if n.endswith(".json"))
print("artifacts:", names)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/exp2_7b_phase0_artifacts"
    os.makedirs(dest, exist_ok=True)
    import shutil
    for n in names:
        shutil.copy2(f"{SRC}/{n}", f"{dest}/{n}")
    print("copied to Drive:", dest)
except Exception as exc:
    print("Drive mount unavailable:", exc)
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w") as tar:
        for n in names:
            tar.add(f"{SRC}/{n}", arcname=n)
    b64 = base64.b64encode(gzip.compress(buf.getvalue(), 9)).decode()
    print("BEGIN_ARTIFACTS_B64")
    for i in range(0, len(b64), 200):
        print(b64[i:i+200])
    print("END_ARTIFACTS_B64")
print("commit these into experiment 2/data/ from the Mac afterwards.")
